In [1]:
import deviantart
import requests, threading
import random
import sqlite3
from bs4 import BeautifulSoup
import json
import time
import requests.auth
import datetime
from requests_oauthlib import OAuth2Session
from oauthlib.oauth2 import BackendApplicationClient
import threading
import os, re
import pandas as pd
from requests.exceptions import HTTPError
import gc

In [10]:
class DeviantArtWatchersFriends:

    def __init__(self, client_id, client_secret, TOKEN_URL, REDIRECT_URI):
        self.client_id = client_id
        self.client_secret = client_secret
        self.TOKEN_URL = TOKEN_URL
        self.REDIRECT_URI = REDIRECT_URI
        self.last_token_refresh_time = 0
        self.token_lock = threading.Lock()
        
    def get_token(self):
        client = BackendApplicationClient(client_id=client_id)
        scope = ['basic', 'user', 'browse']
        oauth = OAuth2Session(client=client, scope=scope, redirect_uri=REDIRECT_URI)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        self.access_token = token['access_token']
        return self.access_token

    # Token refresh function
    token_lock = threading.Lock()  # Create a lock

    def refresh_token(self):
        with self.token_lock:  # Acquire the lock
            if time.time() - self.last_token_refresh_time > 20 * 60:  # Check if token has expired
                self.access_token = self.get_token()
                self.last_token_refresh_time = time.time()  # Update refresh time
                print("Token refreshed.")

    
    def get_response_rate(self, response):
        if response.status_code == 200:
            return json.loads(response.content.decode('utf-8'))
        elif response.status_code == 404:
            return 'user_done'
        elif response.status_code == 429:
            return 'too_many_requests'
        elif response.status_code == 500:
            return 'server error'
        elif response.status_code == 401:
            return 'get new token'

    # Function to get friends
    def get_friends_watchers(self, username, page):
        headers = {"Authorization": f"Bearer {self.access_token}"}
        params_gallery = {"username": username}
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:
                api_url_friends = f"https://www.deviantart.com/api/v1/oauth2/user/friends/{username}?access_token={self.access_token}"
                api_url_watchers = f"https://www.deviantart.com/api/v1/oauth2/user/watchers/{username}?access_token={self.access_token}"
                
                response_watchers = requests.get(api_url_watchers, params={'offset': page, 'limit': 50})
                response_friends = requests.get(api_url_friends, params={'offset': page, 'limit': 50})

                return (response_watchers, response_friends)

            except requests.exceptions.RequestException as e:
                print(f"Error getting info: {e}")
                return None

    # Function to parse friends
    def parse_friends(self, friends):
        users = pd.DataFrame()
        # print(friends.keys())
        # next_offset=friends['next_offset']
        has_more = friends.get('has_more')
        for i in friends['results']:
            a = {'Friends name': i['user']['username'],
                 'user_icon': i['user']['usericon'],
                 'type': i['user']['type'],
                 'is_watching': i['is_watching'],
                 'last_visit': i['lastvisit'],
                 'friends': i['watch']['friend'],
                 'deviations': i['watch']['deviations'],
                 'journals': i['watch']['journals'],
                 'forum_threads': i['watch']['forum_threads'],
                 'critiques': i['watch']['critiques'],
                 'scraps': i['watch']['scraps'],
                 'activity': i['watch']['activity'],
                 'collections': i['watch']['collections']}
            dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
            users = pd.concat([users, dict_pd])
        return (has_more, users)

    def parse_watchers(self, watchers):
        users = pd.DataFrame()
        # print(friends.keys())
        # next_offset=friends['next_offset']
        has_more = watchers.get('has_more')
        for i in watchers['results']:
            a = {'Watchers name': i['user']['username'],
                 'user_icon': i['user']['usericon'],
                 'type': i['user']['type'],
                 'is_watching': i['is_watching'],
                 'last_visit': i['lastvisit'],
                 'activity': i['watch']['activity'],
                 'collections': i['watch']['collections'],
                 'critiques': i['watch']['critiques'],
                 'deviations': i['watch']['deviations'],
                 'forum_threads': i['watch']['forum_threads'],
                 'friend': i['watch']['friend'],
                 'journals': i['watch']['journals'],
                 'scraps': i['watch']['scraps']}
            dict_pd = pd.DataFrame.from_dict(a, orient='index').T
            users = pd.concat([users, dict_pd], ignore_index=True)
        return (has_more, users)

        
    def watchers_friends_data(self, username):
        """Gathers watchers and watching using API."""

        watchers_pd = pd.DataFrame()
        friends_pd = pd.DataFrame()
        has_more = True
        try:
                # Get the initial batch of watchers
                for i in range(0, 5):
                    if has_more == True:
                        resp_watchers, resp_friends = self.get_friends_watchers(username, i)
                        watchers = self.get_response_rate(resp_watchers)
                        friends = self.get_response_rate(resp_friends)
                        if watchers is not None:
                            has_more, parsed_watchers = self.parse_watchers(watchers)
                            if len(parsed_watchers) > 0:
                                parsed_watchers["Deviant"] = username
                                watchers_pd = pd.concat([watchers_pd, parsed_watchers])
                        else: 
                            print(f"No watchers found for {username}")
                            return []
                    
                        if friends is not None:
                            has_more, parsed_frnds = self.parse_friends(friends)
                            if len(parsed_frnds) > 0:
                                parsed_frnds["Deviant"] = username
                                friends_pd = pd.concat([friends_pd, parsed_frnds])
                        else: 
                            print(f"No friends found for {username}")
                            return []




        except requests.exceptions.RequestException as e:
                print(f"Error scraping DeviantArt watchers, friends because of: {e}")
                return [], []
        except Exception as e:
            print(f"Error fetching friends or watchers for {username}: {e}")
            return [], []

        return watchers_pd, friends_pd
    def append_unique_usernames(self, df1, df2, otpt_df):
        """
        Compares usernames in three DataFrames and appends unique usernames to a separate DataFrame.
    
        Args:
            df1, df2, df3: The three DataFrames to compare.
            output_df: The DataFrame to append the unique usernames to.
        """
    
        # Combine usernames from all three DataFrames
        all_usernames = pd.concat([df1["Deviant"], df2["Deviant"]])
    
        # Get unique usernames
        unique_usernames = all_usernames.unique()
    
        # Create a DataFrame for unique usernames
        unique_usernames_df = pd.DataFrame({"unique_dev": unique_usernames})
    
        # Append unique usernames to output_df
        otpt_df = pd.concat([otpt_df, unique_usernames_df], ignore_index=True)
    
        return otpt_df

    def gather_watchers_friends_new(self):
        #current_tag = start_tag
        visited_deviants = set()
        deviant_count = 0  # Keep track of processed deviants
        # Establish a single database connection outside the loop
        #count_for_reauth = 0
        deviant_batch = []
        user_name = []
        #watchers = pd.read_csv("/mnt/hdd/maittewa/deviants_watchersRndmWalk03_2.csv")
        #friends = pd.read_csv("/mnt/hdd/maittewa/deviants_friendsRndmWalk03_2.csv")
        unique_deviants = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_metaDataSnwBall/uniqueDev_metaData_SnwBall_02.csv.gz") #pd.DataFrame()
        #unique_deviants_to_gather = self.append_unique_usernames(watchers,friends,unique_deviants)
        unique_dev_list = set(unique_deviants["Author_Name"].tolist()) #unique_deviants_to_gather["unique_dev"].tolist()
        dev_watcrs = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_watchersSnwball_4/metaDev_watchers-10-07-2025.csv.gz"
        dev_frnds = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_friendsSnwBall_1-4/metaDev_friends-10-07-2025.csv.gz"
        try:
            for deviant in unique_dev_list:
                if deviant not in visited_deviants:
                    deviant_count += 1
                    visited_deviants.add(deviant)
                    print(f"Fetching data for unique deviant: {deviant}")
                    watchers_friends = self.watchers_friends_data(deviant) # Call the corrected watchers_friends_data
                    if len(watchers_friends) == 2:
                        deviant_watchers, deviant_friends = watchers_friends
                    else:
                        print(f"No watchers, friends of {deviant} available")

                    if deviant_watchers is not None and not isinstance(deviant_watchers, list) and not deviant_watchers.empty:  # Save only if not None, not a list and not empty:
                            print(f"fetched deviant watchers for {deviant}")
                            deviant_watchers.to_csv(dev_watcrs, mode="a", header=not os.path.exists(dev_watcrs), index=False)
                            print(f"Saved {deviant} watchers")
                            time.sleep(5) # Add delay after saving watchers
                    else:
                            print(f"Empty deviant watchers for {deviant}")

                            continue

                    if deviant_friends is not None and not isinstance(deviant_friends, list) and not deviant_friends.empty:  # Save only if not None, not a list and not empty:
                            print(f"fetched deviant friends for {deviant}")
                            deviant_friends.to_csv(dev_frnds, mode="a", header=not os.path.exists(dev_frnds), index=False)
                            print(f"Saved {deviant} friends")
                            time.sleep(5) # Add delay after saving friends
                    else:
                            print(f"Empty deviant friends for {deviant}")
                           # Clear the list for the next batch
                            user_name = []
                            time.sleep(5) # Add delay even if friends are empty
                    self.refresh_token()
                    time.sleep(random.uniform(10, 20)) # Add a significant delay between processing deviants

                else:
                            print(f"Skipping already visited {deviant}")

        except requests.exceptions.RequestException as e:
            print(f"Exception {e} occurred")



        print("Gathering of watchers and friends completed.")

    
    def gather_watchers_friends(self):
        """Executes the random walk algorithm."""
        #current_tag = start_tag
        visited_deviants = set()
        unique_deviants = pd.read_csv("/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_metaDataSnwBall/uniqueDev_metaData_SnwBall_02.csv.gz") #pd.DataFrame()
        #unique_deviants_to_gather = self.append_unique_usernames(watchers,friends,unique_deviants)
        unique_dev_list = set(unique_deviants["Author_Name"].tolist()) #unique_deviants_to_gather["unique_dev"].tolist()
        dev_watcrs = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_watchersSnwball_4/metaDev_watchers-10-07-2025.csv.gz"
        dev_frnds = "/mnt/hdd/maittewa/deviantArt_DeviantData/deviantArt_snwBall_fin/deviants_friendsSnwBall_1-4/metaDev_friends-10-07-2025.csv.gz"
        watchers = pd.read_csv(dev_watcrs) #"/mnt/hdd/maittewa/deviants_watchersRndmWalk03_2.csv"
        friends = pd.read_csv(dev_frnds) #"/mnt/hdd/maittewa/deviants_friendsRndmWalk03_2.csv")        
        deviant_count = 0  # Keep track of processed deviants
        deviant_batch = []
        user_name = [] 
        date_visited = set()
        
        try:
            for deviant in unique_dev_list:
                if deviant not in visited_deviants:
                    visited_deviants.add(deviant)
                    deviant_count += 1
                    print(f"Fetching data for unique deviant: {deviant}") 
                    watchers_friends = self.watchers_friends_data(deviant)
                    time.sleep(1) 
                    if len(watchers_friends) == 2:
                        deviant_watchers, deviant_friends = watchers_friends
                    else:
                        print(f"No watchers, friends of {deviant} available")  
                        
                    if deviant_watchers is not None and not isinstance(deviant_watchers, list) and not deviant_watchers.empty:  # Save only if not None, not a list and not empty:
                            print(f"fetched deviant watchers for {deviant}")
                            deviant_watchers.to_csv(dev_watcrs, mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviants_watchersSnwball.csv"), index=False)
                            print(f"Saved {deviant} watchers")
                    else:
                            print(f"Empty deviant watchers for {deviant}")
                            continue
                                
                    if deviant_friends is not None and not isinstance(deviant_friends, list) and not deviant_friends.empty:  # Save only if not None, not a list and not empty:
                            print(f"fetched deviant friends for {deviant}")
                            deviant_friends.to_csv(dev_frnds, mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviants_watchersSnwball.csv"), index=False)
                            print(f"Saved {deviant} friends")
                    else:
                            print(f"Empty deviant friends for {deviant}")
                            time.sleep(5) 
                        
                    self.refresh_token()

                            
                else:
                            print(f"Skipping already visited {deviant}")

        except requests.exceptions.RequestException as e:
            print(f"Exception {e} occurred")

 
                    
        print("Random walk completed.")

In [11]:
# Example usage:
client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"
TOKEN_URL = "https://www.deviantart.com/oauth2/token"
REDIRECT_URI = "https://www.deviantart.com/oauth2/authorize"

In [13]:
# Initialize token refresh timer
random_walk = DeviantArtWatchersFriends(client_id, client_secret, TOKEN_URL, REDIRECT_URI)
random_walk.get_token()
random_walk.refresh_token()
random_walk.gather_watchers_friends()

Token refreshed.
Fetching data for unique deviant: mosquito77
Error fetching friends or watchers for mosquito77: 'str' object has no attribute 'get'
Empty deviant watchers for mosquito77
Fetching data for unique deviant: AXNLphotography
Error fetching friends or watchers for AXNLphotography: 'str' object has no attribute 'get'
Empty deviant watchers for AXNLphotography
Fetching data for unique deviant: ArtbyBeans
Error fetching friends or watchers for ArtbyBeans: 'str' object has no attribute 'get'
Empty deviant watchers for ArtbyBeans
Fetching data for unique deviant: SimplyArtbyJonardT
Error fetching friends or watchers for SimplyArtbyJonardT: 'str' object has no attribute 'get'
Empty deviant watchers for SimplyArtbyJonardT
Fetching data for unique deviant: lariel-istime
Error fetching friends or watchers for lariel-istime: 'str' object has no attribute 'get'
Empty deviant watchers for lariel-istime
Fetching data for unique deviant: LeLovelyZ
Error fetching friends or watchers for Le

KeyboardInterrupt: 

In [ ]:
import pandas as pd
pd

In [ ]:
import pandas as pd
gall1 = pd.read_csv("/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv", on_bad_lines='skip')
#gall2 = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")
#df_meta = pd.read_csv("/mnt/hdd/maittewa/deviants_deviationMetaDataRandomWalkerSince2003.csv")
#df_meta_unique = df_meta.drop_duplicates(subset=['Deviation_id'], keep='first', inplace = False)
#df_meta_unique.to_csv("/mnt/hdd/maittewa/uniqueDeviant_deviationMetaDataRandomWalkerSince2003.csv")

In [ ]:
#df = pd.read_csv("/mnt/hdd/maittewa/uniqueDev_gall_RandomWalkSince2003.csv")
#df = pd.read_csv("/mnt/hdd/maittewa/deviants_deviationMetaDataRandomWalkerSince2003.csv")
gall1.nunique

In [15]:
##One that was running but printing too many times the same thing
def get_random_deviants_from_daily_deviations(self, num_deviants, date):
        """Fetches a list of random deviants from a specific tag and page."""
        # Check if we have a valid token
        url = f"https://www.deviantart.com/api/v1/oauth2/browse/dailydeviations?access_token={self.access_token}"  
        params = {
        "client_id": client_id,
        "client_secret": client_secret,
        "date": date.strftime("%Y-%m-%d"),  # Format date as YYYY-MM-DD
        # ... (other parameters if needed, e.g., limit, offset) ...
        }
        try:
            response = requests.get(url, params=params)
            response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
            data = response.json() 
            #results = self.da.browse_dailydeviations()  
            #results = self.da.browse(tag=tag, offset=(page - 1) * 24)  # 24 results per page by default
            deviants = [deviation["author"]["username"] for deviation in data["results"]]
            random_deviants = random.sample(deviants, min(num_deviants, len(deviants))) 
            return random_deviants
            # Get as many as possible
        except Exception as e:
            print(f"Error fetching deviants: {e}")
            self.refresh_token()
            self.get_random_deviants_from_daily_deviations(num_deviants,date)
    def fetch_gallery_deviationids_metaData(self):
        """
        Executes the random walk algorithm to gather gallery info and metadata.
        """
        # 1. Load Existing Data and Visited Deviants:
        visited_deviants = set()
        already_saved_deviants = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")
        gallery_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv"
        metadata_path = "/mnt/hdd/maittewa/uniqueDeviant_deviationMetaDataRandomWalkerSince2003.csv"

        # Find the bad lines in gallery_data_path
        bad_lines_gallery = find_bad_lines(gallery_data_path)

        # Load the list of existing authors from gallery_data_path into visited_deviants
        if os.path.exists(gallery_data_path):
            print(f"Loading visited deviants from {gallery_data_path}...")
            for chunk in pd.read_csv(gallery_data_path, chunksize=100000, header=0, usecols=['Author_name'], on_bad_lines='skip'):
                # Filter out the rows with bad lines numbers
                if bad_lines_gallery: # only filter if we have bad lines
                    chunk = chunk[~chunk.index.isin(bad_lines_gallery)]
                newly_visited = set(chunk['Author_name'].tolist())
                visited_deviants.update(newly_visited)
                print(f"  Added {len(newly_visited)} deviants from chunk to visited_deviants. Total: {len(visited_deviants)}")
            print(f"Finished loading visited deviants from {gallery_data_path}. Total: {len(visited_deviants)}")

        deviant_count = 0  # Keep track of processed deviants
        save_interval = 5  # Save data after processing this many deviants
        batch_size = 10
        chunk_size = 50  # Adjust chunk size as needed for metadata retrieval
        chunk_size_csv = 10000 # Adjust chunk size as needed for csv retrieval

        # Main execution
        try:
            for i in range(0, len(already_saved_deviants), batch_size):
                batch_usernames = already_saved_deviants['user'][i:i + batch_size].tolist()
                for deviant in batch_usernames:
                    if deviant not in visited_deviants:
                        visited_deviants.add(deviant)
                        deviant_count += 1
                        deviant_gall_data = pd.DataFrame(columns=['Author_name', 'Deviation_id'])
                        meta = pd.DataFrame()
                        print(f"Gathering gallery info and metadata for {deviant}, count {deviant_count}")
                        gallery = self.get_gallery(deviant)
                        if gallery is not None:
                            gallery_df = pd.DataFrame(gallery)
                            #Ensure the column name is consistent
                            if 'deviation_id' in gallery_df.columns:
                                gallery_df.rename(columns={'deviation_id': 'Deviation_id'}, inplace=True)
                            
                            deviant_gall_data = pd.concat([deviant_gall_data,gallery_df], ignore_index=True)
                            print(f'Gathered and parsed gallery data for {deviant}')

                        else:
                            print(f"No gallery data of {deviant} available")
                        time.sleep(random.uniform(1,2))
                        if deviant_gall_data is not None and 'Deviation_id' in deviant_gall_data.columns:
                            devIds = deviant_gall_data["Deviation_id"].tolist()
                            metadata_chunks = [devIds[i:i + chunk_size] for i in range(0, len(devIds), chunk_size)]
                            for chunk in metadata_chunks:
                                metadata = self.get_metadata(chunk)
                                if metadata:
                                    parsed_df = self.parse_metadata(metadata)
                                    meta = pd.concat([meta, parsed_df], ignore_index=True)  # Append parsed data
                                print(f'Gathered and parsed deviation metadata for {deviant}')
                                # 3. Save Data in Batches:
                                if deviant_count % save_interval == 0:
                                # Append to CSV with `mode="a"`
                                    deviant_gall_data.to_csv(gallery_data_path, mode="a", header=not os.path.exists(gallery_data_path), index=False)
                                    meta.to_csv(metadata_path, mode="a", header=not os.path.exists(metadata_path), index=False)

                                    print(f"      Saved data for {deviant_count} deviants.")

                            # Delete only if not saving
                            if deviant_count % save_interval != 0:
                                del deviant_gall_data
                                del meta
                                gc.collect()

            else:
                print(f"The path {already_saved_deviants_path} does not exists")
        except requests.exceptions.RequestException as e:
            print(f"Exception {e} occurred")
        except pd.errors.ParserError as e:
            print(f"A parser error occurred: {e}")

        print("Random walk completed.")

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 26)

In [ ]:
###For reducing the too many print statements
def fetch_gallery_deviationids_metaData(self):
        """
        Executes the random walk algorithm to gather gallery info and metadata.
        """
        # 1. Load Existing Data and Visited Deviants:
        visited_deviants = set()
        already_saved_deviants = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")
        gallery_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv"
        metadata_path = "/mnt/hdd/maittewa/uniqueDeviant_deviationMetaDataRandomWalkerSince2003.csv"

        # Find the bad lines in gallery_data_path
        bad_lines_gallery = self.find_bad_lines(gallery_data_path)

        # Load the list of existing authors from gallery_data_path into visited_deviants
        if os.path.exists(gallery_data_path):
            print(f"Loading visited deviants from {gallery_data_path}...")
            for chunk in pd.read_csv(gallery_data_path, chunksize=10000, header=0, usecols=['Author_name'], on_bad_lines='skip'):
                # Filter out the rows with bad lines numbers
                if bad_lines_gallery: # only filter if we have bad lines
                    chunk = chunk[~chunk.index.isin(bad_lines_gallery)]
                newly_visited = set(chunk['Author_name'].tolist())
                visited_deviants.update(newly_visited)
                print(f"  Added {len(newly_visited)} deviants from chunk to visited_deviants. Total: {len(visited_deviants)}")
            print(f"Finished loading visited deviants from {gallery_data_path}. Total: {len(visited_deviants)}")

        deviant_count = 0  # Keep track of processed deviants
        save_interval = 5  # Save data after processing this many deviants
        batch_size = 10
        chunk_size = 50  # Adjust chunk size as needed for metadata retrieval
        chunk_size_csv = 10000 # Adjust chunk size as needed for csv retrieval
        deviant_gall_data = pd.DataFrame()
        meta = pd.DataFrame()

        # Main execution
        try:
            for i in range(0, len(already_saved_deviants), batch_size):
                batch_usernames = already_saved_deviants['user'][i:i + batch_size].tolist()

                for deviant in batch_usernames:
                    if deviant not in visited_deviants:
                        visited_deviants.add(deviant)
                        deviant_count += 1
                        print(f"Gathering gallery info and metadata for {deviant}")

                        gallery = self.get_gallery(deviant)
                        if gallery is not None:
                            gallery_df = pd.DataFrame(gallery)
                            deviant_gall_data = pd.concat([deviant_gall_data, gallery_df], ignore_index=True)
                            print(f'Gathered and parsed gallery data for {deviant}')
                            # ---------------------------------------------------------------
                            # All metadata for a user should be gathered here
                            devIds = deviant_gall_data["Deviation_id"].tolist()
                            metadata_chunks = [devIds[j:j + chunk_size] for j in range(0, len(devIds), chunk_size)]
                            for chunk in metadata_chunks:
                                metadata = self.get_metadata(chunk)
                                if metadata:
                                    parsed_df = self.parse_metadata(metadata)
                                    meta = pd.concat([meta, parsed_df], ignore_index=True)

                            print(f'Gathered and parsed deviation metadata for {deviant}')

                            # ---------------------------------------------------------------
                        else:
                            print(f"No gallery data of {deviant} available")

                        if deviant_count % save_interval == 0:
                            # Save gallery data in chunks
                            for start in range(0, len(deviant_gall_data), chunk_size_csv):
                                end = start + chunk_size_csv
                                chunk_to_save = deviant_gall_data.iloc[start:end]
                                chunk_to_save.to_csv(gallery_data_path, mode='a', header=not os.path.exists(gallery_data_path), index=False)
                            deviant_gall_data = pd.DataFrame()  # Reset for the next batch

                            # Save metadata in chunks
                            for start in range(0, len(meta), chunk_size_csv):
                                end = start + chunk_size_csv
                                chunk_to_save = meta.iloc[start:end]
                                chunk_to_save.to_csv(metadata_path, mode='a', header=not os.path.exists(metadata_path), index=False)
                            meta = pd.DataFrame()  # Reset for the next batch

                            print(f"Saved data for {deviant_count} deviants.")

                        self.refresh_token()
                        time.sleep(random.uniform(1, 2))
                    else:
                        print(f"Skipping already visited {deviant}")

        except RequestException as e:
            print(f"Request Exception: {e}")

        except Exception as e:
            print(f"An unexpected error occurred: {e}")

        finally:
            # Save any remaining data
            if not deviant_gall_data.empty:
                for start in range(0, len(deviant_gall_data), chunk_size_csv):
                    end = start + chunk_size_csv
                    chunk_to_save = deviant_gall_data.iloc[start:end]
                    chunk_to_save.to_csv(gallery_data_path, mode='a', header=not os.path.exists(gallery_data_path), index=False)
            if not meta.empty:
                for start in range(0, len(meta), chunk_size_csv):
                    end = start + chunk_size_csv
                    chunk_to_save = meta.iloc[start:end]
                    chunk_to_save.to_csv(metadata_path, mode='a', header=not os.path.exists(metadata_path), index=False)

        print("Random walk completed.")

In [ ]:
def fetch_gallery_deviationids_metaData(self):
        """Executes the random walk algorithm."""
        # 1. Load Existing Data and Visited Deviants:
        visited_deviants = set()
        already_saved_deviants = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")

        gallery_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv"
        #metadata_path = "/mnt/hdd/maittewa/uniqueDeviant_deviationMetaDataRandomWalkerSince2003.csv"

        if os.path.exists(gallery_data_path):
            existing_gallery_df = pd.read_csv(gallery_data_path)
            visited_deviants.update(set(existing_gallery_df['Author_name'].tolist()))
        
        deviant_count = 0  # Keep track of processed deviants
        # Establish a single database connection outside the loop
        #count_for_reauth = 0
        deviant_batch = []
        user_name = [] 
        deviant_gall_data = pd.DataFrame()
        meta = pd.DataFrame()
        save_interval = 10  # Save data after processing this many deviants
        deviant_count = 0
        batch_size = 10

        # Main execution
        # Split the deviation IDs into chunks to avoid exceeding API limits
        try:
            for i in range(0, len(already_saved_deviants), batch_size):
                batch_usernames = already_saved_deviants['user'][i:i + batch_size].tolist()

                for deviant in batch_usernames:
                    if deviant not in visited_deviants:
                        visited_deviants.add(deviant)
                        deviant_count += 1
                        print(f"Gathering gallery info and metadata for {deviant}")
                        gallery = self.get_gallery(deviant)
                        if gallery is not None:
                            gallery_df = pd.DataFrame(gallery)
                            deviant_gall_data = pd.concat([deviant_gall_data,gallery_df])
                            print(f'Gathered and parsed gallery data for {deviant}')
                            
                        else:
                            print(f"No gallery data of {deviant} available")
                        time.sleep(random.uniform(1,2)) 
                        if deviant_gall_data is not None:
                            devIds = deviant_gall_data["Deviation_id"].tolist()     
                            metadata_chunks = [devIds[i:i + chunk_size] for i in range(0, len(devIds), chunk_size)]
                            for chunk in metadata_chunks:
                                metadata = self.get_metadata(chunk)
                            if metadata:
                                parsed_df = self.parse_metadata(metadata)
                                meta = pd.concat([meta, parsed_df], ignore_index=True)  # Append parsed data
                            print(f'Gathered and parsed deviation metadata for {deviant}')
                            # 3. Save Data in Batches:
                            if deviant_count % save_interval == 0:
                            # Append to CSV with `mode="a"`
                                deviant_gall_data.to_csv(gallery_data_path, mode="a", header=not os.path.exists(gallery_data_path), index=False)
                                #meta.to_csv(metadata_path, mode="a", header=not os.path.exists(metadata_path), index=False)

                                # Reset DataFrames for the next batch
                                deviant_gall_data = pd.DataFrame()
                                #syyyyyyyyyyyyyxmeta = pd.DataFrame()
                                print(f"Saved data for {deviant_count} deviants.")

                            self.refresh_token()
                           # Clear the list for the next batch
                           # time.sleep(5) 
                            
                    else:
                        print(f"Skipping already visited {deviant}")

        except requests.exceptions.RequestException as e:
            print(f"Exception {e} occurred")
            
        print("Random walk completed.")

In [9]:

    def fetch_gallery_deviationids_metaData(self):
        """
        Executes the random walk algorithm to gather gallery info and metadata.
        """
        # 1. Load Existing Data and Visited Deviants:
        visited_deviants = set()
        already_saved_deviants_path = "/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv"
        gallery_data_path = "/mnt/hdd/maittewa/uniqueDev_gall_RndmWalk03_4.csv"
        metadata_path = "/mnt/hdd/maittewa/uniqueDev_dvtnMetaRndmWalk03_1.csv"

        if os.path.exists(gallery_data_path):
            existing_gallery_df = pd.read_csv(gallery_data_path)
            visited_deviants.update(set(existing_gallery_df['Author_name'].tolist()))

        deviant_count = 0  # Keep track of processed deviants
        save_interval = 5  # Save data after processing this many deviants
        batch_size = 10
        chunk_size = 50  # Adjust chunk size as needed for metadata retrieval
        chunk_size_csv = 1000 # Adjust chunk size as needed for csv retrieval
        # Find the bad lines
        bad_lines = find_bad_lines(gallery_data_path)
        bad_lines_numbers = {line_number for line_number, _ in bad_lines}
        #Main execution
       #Main execution
        try:
            # Check if the file exists
            if os.path.exists(already_saved_deviants_path):
                # Iterate over the file in chunks
                # In this line we read the already_saved_deviants_path file
                for chunk in pd.read_csv(already_saved_deviants_path, chunksize=chunk_size_csv, header=None, on_bad_lines="skip"):

                    # Iterate over each row of the current chunk to get the usernames
                    for index, row in chunk.iterrows():

                        deviant = row['user']

                        # check that the deviant is a string and not a number
                        if type(deviant) is not str:
                            continue

                        if deviant not in visited_deviants:
                            deviant_gall_data = pd.DataFrame(columns=['Author_name', 'Deviation_id'])
                            meta = pd.DataFrame()
                            visited_deviants.add(deviant)
                            deviant_count += 1
                            print(f"Gathering gallery info and metadata for {deviant}, count {deviant_count}")
                            gallery = self.get_gallery(deviant)
                            if gallery is not None:
                                gallery_df = pd.DataFrame(gallery)
                                #Ensure the column name is consistent
                                if 'deviation_id' in gallery_df.columns:
                                    gallery_df.rename(columns={'deviation_id': 'Deviation_id'}, inplace=True)

                                deviant_gall_data = pd.concat([deviant_gall_data,gallery_df], ignore_index=True)
                                print(f'Gathered and parsed gallery data for {deviant}')

                            else:
                                print(f"No gallery data of {deviant} available")
                            time.sleep(random.uniform(1,2))
                            if deviant_gall_data is not None and 'Deviation_id' in deviant_gall_data.columns:
                                devIds = deviant_gall_data["Deviation_id"].tolist()
                                metadata_chunks = [devIds[i:i + chunk_size] for i in range(0, len(devIds), chunk_size)]
                                for chunk in metadata_chunks:
                                    metadata = self.get_metadata(chunk)
                                    if metadata:
                                        parsed_df = self.parse_metadata(metadata)
                                        meta = pd.concat([meta, parsed_df], ignore_index=True)  # Append parsed data
                                print(f'Gathered and parsed deviation metadata for {deviant}')
                                # 3. Save Data in Batches:
                                if deviant_count % save_interval == 0:
                                # Append to CSV with `mode="a"`
                                    deviant_gall_data.to_csv(gallery_data_path, mode="a", header=not os.path.exists(gallery_data_path), index=False)
                                    meta.to_csv(metadata_path, mode="a", header=not os.path.exists(metadata_path), index=False)

                                    print(f"Saved data for {deviant_count} deviants.")
                            
                            # Delete only if not saving
                            if deviant_count % save_interval != 0:
                                del deviant_gall_data
                                del meta
                                gc.collect()
                        else:
                            print(f"Skipping already visited {deviant}")

            else:
              print(f"The path {already_saved_deviants_path} does not exists")
        except requests.exceptions.RequestException as e:
            print(f"Exception {e} occurred")

        print("Random walk completed.")



In [10]:
#Actual random Walk for gathering deviant data
def run_random_walk(self, num_devs, num_dates):
        """Executes the random walk algorithm."""
        #current_tag = start_tag
        visited_deviants = set()
        dev_df = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")
        already_saved_deviants = set(dev_df['user'].tolist()) 
        deviant_count = 0  # Keep track of processed deviants
        # Establish a single database connection outside the loop
        #count_for_reauth = 0
        deviant_batch = []
        user_name = [] #, user_watchers, user_friends, deviatn_metadata = [], [], [], [] #user_profile,
        # Main execution
        start_date = datetime.date(2003, 1, 1)  # 21 years ago
        end_date = datetime.date.today()
        date_visited = set()
        
        try:
            for _ in range(num_dates):
                random_date = start_date + datetime.timedelta(days=random.randint(0, (end_date - start_date).days))

                random_deviants = self.get_random_deviants_from_daily_deviations(num_devs,random_date)
                if random_deviants is not None:
                    for deviant in random_deviants:
                        if deviant not in visited_deviants and deviant not in already_saved_deviants:
                            visited_deviants.add(deviant)
                            print(f"Fetching data for unique deviant: {deviant} on date {random_date}") 
                            profile = self.get_profile(deviant)
                            deviant_profile = self.parse_user_profile(profile)
                            time.sleep(random.uniform(1,5))                   
                            watchers_friends_gallery = self.watchers_friends_gallery_data(deviant)
                            if len(watchers_friends_gallery) == 3:
                                deviant_watchers, deviant_friends, deviant_gallery_data = watchers_friends_gallery
                            else:
                                print(f"No watchers, friends or gallery data of {deviant} available")
                            time.sleep(random.uniform(1,5))    
                            devIds = deviant_gallery_data["Deviation_Id"].tolist()
                            #deviant_deviations = self.download_deviations(deviant)
                            #print(f"Downloaded {deviant}'s gallery")
                            #time.sleep(random.uniform(1,5))                   
                            deviations_metadata = self.parse_metadata(devIds)
                            print(f'Parsed deviation metadata for {deviant}')
                            time.sleep(random.uniform(1,5))                   
                            #print (deviations_metadata)
                            user_name.append(deviant)
                            deviant_df = pd.DataFrame(user_name, columns=["deviant_username"])
                            deviant_df.to_csv("/mnt/hdd/maittewa/deviantsRandomWalkerSince2003.csv", mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviantsRandomWalkerSince2003.csv"))
                            print(f"Saved {deviant} name")
                            if deviant_profile is not None:
                                print(f"fetched user profile data for {deviant}")
                                deviant_profile.to_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv", mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv"), index=False)
                                print(f"Saved {deviant} profile")
                            else:
                                print(f"Empty user profile data for {deviant}")
            
                            if deviant_watchers is not None and not isinstance(deviant_watchers, list) and not deviant_watchers.empty:  # Save only if not None, not a list and not empty:
                                print(f"fetched deviant watchers for {deviant}")
                                deviant_watchers.to_csv("/mnt/hdd/maittewa/deviants_watchersRandomWalkerSince2003.csv", mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviants_watchersRandomWalkerSince2003.csv"), index=False)
                                print(f"Saved {deviant} watchers")
                            else:
                                print(f"Empty deviant watchers for {deviant}")
                                
                            if deviant_friends is not None and not isinstance(deviant_friends, list) and not deviant_friends.empty:  # Save only if not None, not a list and not empty:
                                print(f"fetched deviant friends for {deviant}")
                                deviant_friends.to_csv("/mnt/hdd/maittewa/deviants_friendsRandomWalkerSince2003.csv", mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviants_friendsRandomWalkerSince2003.csv"), index=False)
                                print(f"Saved {deviant} friends")
                            else:
                                print(f"Empty deviant friends for {deviant}")
            
                            if deviations_metadata is not None:
                                print(f"fetched deviant deviations metadata for {deviant}")
                                deviations_metadata.to_csv("/mnt/hdd/maittewa/deviatn_metadataRandomWalkerSince2003.csv", mode="a", header=not os.path.exists("/mnt/hdd/maittewa/deviatn_metadataRandomWalkerSince2003.csv"), index=False)
                                print(f"Saved {deviant} deviations metadata")
                            else:
                                print(f"Empty deviations metadata for {deviant}")
                           
                           # Clear the list for the next batch
                            user_name = []
                            time.sleep(5) 
                            
                        else:
                            print(f"Skipping already visited {deviant}")
                else:
                    print(f"No deviant for the date {random_date}")

        except requests.exceptions.RequestException as e:
            print(f"Exception {e} occurred")

        # Dump any remaining data in the batch
        if deviant_batch:
            #self.store_data(deviant, user_profile, deviant_watchers, deviant_friends, deviations_metadata)
            print(f"Dumped remaining data for {len(deviant_batch)} deviations to database.")
 
                    
        print("Random walk completed.")

In [11]:
df_check = pd.read_csv("/mnt/hdd/maittewa/deviants_profileRandomWalkerSince2003.csv")

In [59]:
df_check.user.value_counts()

user
mjdaluz           6
NibelArt          6
Raindropmemory    5
Sieskja           5
Stridsberg        5
                 ..
bia37             1
Leventart         1
TheShanar         1
Alienphysique     1
Heyriel           1
Name: count, Length: 21517, dtype: int64

In [102]:
def get_token():
        client = BackendApplicationClient(client_id=client_id)
        oauth = OAuth2Session(client=client)
        post_data = {"grant_type": "client_credentials",
                 "redirect_uri": REDIRECT_URI}
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret, 
                                  data=post_data)
        # Extract the access token
        access_token = token['access_token']
        return access_token
    
def get_response_rate(response):
        if response.status_code == 200:
            return json.loads(response.content.decode('utf-8'))
        elif response.status_code == 404:
            return 'user_done'
        elif response.status_code == 429:
            return 'too_many_requests'
        elif response.status_code == 500:
            return 'server error'
        elif response.status_code == 401:
            return 'get new token'

# Function to get friends
def get_friends(username, page):
    token = get_token()
    api_url = f"https://www.deviantart.com/api/v1/oauth2/user/friends/{username}?access_token={token}"
    response = requests.get(api_url, params={'offset': page, 'limit': 50})
    return response

# Function to get watchers
def get_watchers(username, page):
    token = get_token()
    api_url = f"https://www.deviantart.com/api/v1/oauth2/user/watchers/{username}?access_token={token}"
    response = requests.get(api_url, params={'offset': page, 'limit': 50})
    return response

# Function to parse friends
def parse_friends(friends):
    users = pd.DataFrame()
    # print(friends.keys())
    # next_offset=friends['next_offset']
    has_more = friends.get('has_more')
    for i in friends['results']:
        a = {'username': i['user']['username'],
                 'user_icon': i['user']['usericon'],
                 'type': i['user']['type'],
                 'is_watching': i['is_watching'],
                 'last_visit': i['lastvisit'],
                 'friends': i['watch']['friend'],
                 'deviations': i['watch']['deviations'],
                 'journals': i['watch']['journals'],
                 'forum_threads': i['watch']['forum_threads'],
                 'critiques': i['watch']['critiques'],
                 'scraps': i['watch']['scraps'],
                 'activity': i['watch']['activity'],
                 'collections': i['watch']['collections']}
        dict_pd = pd.DataFrame.from_dict(a, orient='index').transpose()
        users = pd.concat([users, dict_pd])
    return has_more, users

def parse_watchers(watchers):
    users = pd.DataFrame()
    # print(friends.keys())
    # next_offset=friends['next_offset']
    has_more = watchers.get('has_more')
    for i in watchers['results']:
        a = {'username': i['user']['username'],
                 'user_icon': i['user']['usericon'],
                 'type': i['user']['type'],
                 'is_watching': i['is_watching'],
                 'last_visit': i['lastvisit'],
                 'activity': i['watch']['activity'],
                 'collections': i['watch']['collections'],
                 'critiques': i['watch']['critiques'],
                 'deviations': i['watch']['deviations'],
                 'forum_threads': i['watch']['forum_threads'],
                 'friend': i['watch']['friend'],
                 'journals': i['watch']['journals'],
                 'scraps': i['watch']['scraps']}
        dict_pd = pd.DataFrame.from_dict(a, orient='index').T
        users = pd.concat([users, dict_pd], ignore_index=True)
    return has_more, users

In [103]:
"""Gathers watchers and watching using API."""

watchers_pd = pd.DataFrame()
# Get watching (friends)
friends_pd = pd.DataFrame()
has_more = True
username = "damter"
try:
    # Get the initial batch of watchers
    for i in range(0, 5):
        if has_more == True:
            resp = get_watchers(username, i)
            watchers = get_response_rate(resp)
            if watchers is not None:
                has_more, parsed_watchers = parse_watchers(watchers)
                if len(parsed_watchers) > 0:
                    watchers_pd = pd.concat([watchers_pd, parsed_watchers])
                
    # Get the initial batch of watching users
    for i in range(0, 5):
        if has_more == True:
            resp = get_friends(username, i)
            friends = get_response_rate(resp)
            if friends is not None:
                has_more, parsed_frnds = parse_friends(friends)
                if len(parsed_frnds) > 0:
                    friends_pd = pd.concat([friends_pd, parsed_frnds])

except requests.exceptions.RequestException as e:
    print(f"Error scraping DeviantArt watchers and watching: {e}")

In [104]:
friends_pd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


In [1]:
def get_token():
        client = BackendApplicationClient(client_id=client_id)
        oauth = OAuth2Session(client=client)
        token = oauth.fetch_token(token_url=TOKEN_URL, client_id=client_id, client_secret=client_secret)
        # Extract the access token
        access_token = token['access_token']
        return access_token

In [2]:
client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"
TOKEN_URL = "https://www.deviantart.com/oauth2/token"
REDIRECT_URI = "https://www.deviantart.com/oauth2/authorize"

In [6]:
import pandas as pd
import requests
import os

da = deviantart.Api("42096", "97080792c6d30a4178965e41f1ca15de")

# Specify the username
username = "RavenRemo"

# Get the user's gallery
gallery = da.get_gallery_folder(username)
output_dir = f"/mnt/hdd/maittewa/downloaded_gallery_{username}"

all_deviations = []
has_more = True
while has_more:
  for deviation in gallery['results']:
    deviation_data = {}
    if hasattr(deviation, 'title'):
        deviation_data['title'] = deviation.title

    # Get deviation details including tags
    deviation_details = da.get_deviation(deviation.deviationid)
    if hasattr(deviation_details, 'tags'):
      deviation_data['tags'] = deviation_details.tags

    all_deviations.append(deviation_data)

  has_more = gallery['has_more']
  if has_more:
    gallery = da.get_gallery_folder(username, offset=gallery['next_offset'])

    metadata = {
        "title": deviation.title,
        "deviationid": deviation.deviationid,
        "url": deviation.url,
        "category": deviation.category
        # Add other metadata fields as needed
    }
    
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, username + ".json")
    
    with open(filename, "w") as f:
        json.dump(metadata, f, indent=4)
    
    print(f"Saved metadata for deviation {deviation_id} to {filename}")
# Print the list of tags for each deviation
for deviation in all_deviations:
  print(f"Deviation: {deviation['title']}")
  if 'tags' in deviation:
    print(f"Tags: {', '.join(deviation['tags'])}")
  else:
    print("No tags found for this deviation.")
  print("-" * 20)

Saved metadata for deviation 59FAB303-3905-7EDA-438C-F2BDC04FACB7 to /mnt/hdd/maittewa/downloaded_gallery_RavenRemo/RavenRemo.json
Saved metadata for deviation 59FAB303-3905-7EDA-438C-F2BDC04FACB7 to /mnt/hdd/maittewa/downloaded_gallery_RavenRemo/RavenRemo.json
Saved metadata for deviation 59FAB303-3905-7EDA-438C-F2BDC04FACB7 to /mnt/hdd/maittewa/downloaded_gallery_RavenRemo/RavenRemo.json
Saved metadata for deviation 59FAB303-3905-7EDA-438C-F2BDC04FACB7 to /mnt/hdd/maittewa/downloaded_gallery_RavenRemo/RavenRemo.json
Saved metadata for deviation 59FAB303-3905-7EDA-438C-F2BDC04FACB7 to /mnt/hdd/maittewa/downloaded_gallery_RavenRemo/RavenRemo.json
Saved metadata for deviation 59FAB303-3905-7EDA-438C-F2BDC04FACB7 to /mnt/hdd/maittewa/downloaded_gallery_RavenRemo/RavenRemo.json
Saved metadata for deviation 59FAB303-3905-7EDA-438C-F2BDC04FACB7 to /mnt/hdd/maittewa/downloaded_gallery_RavenRemo/RavenRemo.json
Saved metadata for deviation 59FAB303-3905-7EDA-438C-F2BDC04FACB7 to /mnt/hdd/maitt

In [ ]:
 def deviations_metadata(self, deviant):
        # Get the gallery folder
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()
        if self.access_token:
            try:
                gallery = self.da.get_gallery_folder(deviant)
                deviations_metadata = pd.DataFrame()
                for deviation in gallery['results']:
                    metadata = {}
                    metadata = {
                        "deviation_id": deviation.deviationid,
                        "title": deviation.title,
                        "category": deviation.category}
                    if hasattr(deviation, 'content') and deviation.content and 'src' in deviation.content:
                        metadata['image_width'] = deviation.content['width']
                        metadata["image_height"] = deviation.content["height"]
                        #filename = f"{deviation_id}.json"
                        #filepath = os.path.join(output_dir, metadata_folder, username + ".json")
                        #os.makedirs(os.path.dirname(filepath), exist_ok=True)  
                        dict_pd = pd.DataFrame.from_dict(metadata, orient='index').transpose()
                        deviations_metadata = pd.concat([deviations_metadata, dict_pd])
                return deviations_metadata
            except Exception as e:
                print(f"Encountering the following {e} error")
                self.refresh_token()
                self.download_deviations(deviant)


    def download_deviations(self,deviant):  
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()
        self.da.token = self.access_token
        if self.da.token:
            try:
                gallery = self.da.get_gallery_folder(deviant)
                # Create an output directory for downloaded images
                output_dir = f"/mnt/hdd/maittewa/deviantartGallery/downloaded_gallery_{deviant}"
                os.makedirs(output_dir, exist_ok=True)
                #os.makedirs(gallery_folder, exist_ok=True)
                # Download images
                has_more = True
                while has_more:
                    for deviation in gallery['results']:
                        if hasattr(deviation, 'content') and deviation.content:
                            if 'src' in deviation.content:
                                image_url = deviation.content['src']
                                if image_url:
                                    filename = f"{deviation.title}-{deviation.deviationid}.jpg"
                                    filepath = os.path.join(output_dir, deviant, filename)
                                    os.makedirs(os.path.dirname(filepath), exist_ok=True)  
                                    # Download and save the image
                                    # Download and save the image
                                    try:
                                        response = requests.get(image_url, stream=True)  # Assign response
                                        response.raise_for_status() # Check for errors
                            
                                        with open(filepath, 'wb') as image_file:
                                            for chunk in response.iter_content(chunk_size=8192):
                                                image_file.write(chunk)
                                            #print(f"Downloaded: {filename}")
        
                                    except requests.exceptions.RequestException as e:
                                        print(f"Error downloading {image_url}: {e}")
        
                                else:
                                    image_url = None
                                    print(f"Image URL not found for deviation ID: {deviation_id}")
                
                    # Check if there are more deviations to fetch
                    has_more = gallery['has_more']
                    if has_more:
                        gallery = self.da.get_gallery_folder(
                            deviant, offset=gallery['next_offset'])

            except Exception as e:
                print(f"Encountering the following {e} error")
                self.refresh_token()
                self.download_deviations(deviant)

In [ ]:
    def download_deviations(self,deviant):  
        # Check if we have a valid token
        if not self.access_token:
            self.access_token = self.get_token()

        if self.access_token:
            try:
                
                # Create an output directory for downloaded images
                output_dir = f"/mnt/hdd/maittewa/deviantartGallery/downloaded_gallery_{deviant}"
                os.makedirs(output_dir, exist_ok=True)
                #os.makedirs(gallery_folder, exist_ok=True)
                # Download images
                has_more = True
                while has_more:
                    for deviation in gallery['results']:
                        if hasattr(deviation, 'content') and deviation.content:
                            if 'src' in deviation.content:
                                image_url = deviation.content['src']
                                if image_url:
                                    filename = f"{deviation.title}-{deviation.deviationid}.jpg"
                                    filepath = os.path.join(output_dir, deviant, filename)
                                    os.makedirs(os.path.dirname(filepath), exist_ok=True)  
                                    # Download and save the image
                                    # Download and save the image
                                    try:
                                        response = requests.get(image_url, stream=True)  # Assign response
                                        response.raise_for_status() # Check for errors
                            
                                        with open(filepath, 'wb') as image_file:
                                            for chunk in response.iter_content(chunk_size=8192):
                                                image_file.write(chunk)
                                            #print(f"Downloaded: {filename}")
        
                                    except requests.exceptions.RequestException as e:
                                        print(f"Error downloading {image_url}: {e}")
        
                                else:
                                    image_url = None
                                    print(f"Image URL not found for deviation ID: {deviation_id}")
                
                    # Check if there are more deviations to fetch
                    has_more = gallery['has_more']
                    if has_more:
                        gallery = self.da.get_gallery_folder(
                            deviant, offset=gallery['next_offset'])

            except Exception as e:
                print(f"Encountering the following {e} error")
                self.refresh_token()
                self.download_deviations(deviant)



In [12]:
import deviantart
import os

# Replace with your client ID and client secret
client_id = "42096"
client_secret = "97080792c6d30a4178965e41f1ca15de"

# Create an API object
da = deviantart.Api(client_id, client_secret)

# Specify the username of the deviant whose gallery folder you want to download
username = "Ramonn90"  # Example username

# Create an output directory for downloaded images
output_dir = f"/mnt/hdd/maittewa/deviantartGallery/"
gallery_folder = f"downloaded_gallery_{username}"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(gallery_folder, exist_ok=True)

# Get the gallery folder
gallery = da.get_gallery_folder(username)
deviations_metadata = pd.DataFrame()
# Download images
has_more = True
while has_more:
    for deviation in gallery['results']:
        if hasattr(deviation, 'content') and deviation.content:
            if 'src' in deviation.content:
                image_url = deviation.content['src']
                if image_url:
                    filename = f"{deviation.title}-{deviation.deviationid}.jpg"
                    filepath = os.path.join(output_dir, gallery_folder, username, filename)
                    os.makedirs(os.path.dirname(filepath), exist_ok=True)  
                    # Download and save the image
                    try:
                        response = requests.get(image_url, stream=True)  # Assign response
                        response.raise_for_status() # Check for errors
                    
                        with open(filepath, 'wb') as image_file:
                            for chunk in response.iter_content(chunk_size=8192):
                                image_file.write(chunk)
                            print(f"Downloaded: {filename}")

                    except requests.exceptions.RequestException as e:
                         print(f"Error downloading {download_url}: {e}")

                else:
                    image_url = None
                    print(f"Image URL not found for deviation ID: {deviation_id}")

    # Check if there are more deviations to fetch
    has_more = gallery['has_more']
    if has_more:
        gallery = da.get_gallery_folder(
            username, offset=gallery['next_offset'])

Downloaded: Practice - Nov Patreon-965D3F67-522F-F074-0BD6-4011FC43A5F1.jpg
Downloaded: Practice - Nov Patreon-CB7A1582-B8F2-ECCB-50E7-51B916E1976F.jpg
Downloaded: Practice - Nov Patreon-1B48822B-9E90-561E-133B-147A6F197FB6.jpg
Downloaded: Practice - Nov Patreon-3B540F3C-99E0-CEA0-7C70-4B2FBA6AF769.jpg
Downloaded: Practice - Nov Patreon-73312238-050F-51F1-593D-B3273A19E531.jpg
Downloaded: New Brush - Oct Patreon-0DD16D20-D3E8-4056-912D-71E3F3AA0730.jpg
Downloaded: Catcher - Oct Patreon-B83462DC-085E-AE4C-735F-8008963D3449.jpg
Downloaded: Serve - September Patreon-C1702605-3277-F79B-79A9-15F3A8CCCAC6.jpg
Downloaded: Night - Sep Patreon-B6BD114D-E834-4EA1-562B-2C0E19C91AFF.jpg
Downloaded: Whisper - Sep Patreon-367096C0-E0D6-1027-E3B9-C21AD689EA8A.jpg
Downloaded: Gravity-308A1245-0126-D11F-8689-0C5A912150AC.jpg
Downloaded: Hands Holding Tutorial-5B87C790-D646-514D-03E1-7F901BF9B7D9.jpg
Downloaded: Cut-465C167A-D3FB-AB0E-0ACC-7FA6ED12BD1F.jpg
Downloaded: Open-D1D88C2E-B441-12F9-38EB-6D8069

KeyboardInterrupt: 

In [ ]:
#Error handling
#print(f"Entering HTTPError handler...")
                print(f"HTTP Error: {e} for user: {username}")
                status_code_match = re.search(r"(\d{3})", str(e))  # Extract status code from error message
                if status_code_match and status_code_match.group(1) == "401":
                    print("401 Unauthorized error.")
                else:
                    # Handle other HTTP errors
                    print(f"Handling other HTTP errors...")

###Profile api response 
if profile_response is None:
            print("Status code is None")
        elif profile_response.status_code != 200:
            print(f"API request failed with status code {s.status_code}: {s.text}")
        else:
            # API request was successful
            d = json.loads(profile_response.text)
                